In [ ]:
# Reading from Event Hubs via its Kafka endpoint: no extra library is needed on DBR 18 (Spark 4.1 / Scala 2.13).
# azure-eventhub is only required by the producer script that sends data, not by this notebook.

In [ ]:
eh_namespace = "eventhubairline2"
eh_name = "airline"   # must match the event hub the producer script sends to

# Preferably: event_hub_connection_string = dbutils.secrets.get("<scope>", "eventhub-conn-str")
event_hub_connection_string = "Endpoint=sb://eventhubairline2.servicebus.windows.net/;SharedAccessKeyName=RootManageSharedAccessKey;SharedAccessKey=<YOUR_EVENT_HUB_KEY>"

catalog = "airline_cluster"
schema_name = "airline"

# Volume used for all streaming checkpoints (display() and writeStream need an explicit location)
spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog}.{schema_name}.checkpoints")
checkpoint_base = f"/Volumes/{catalog}/{schema_name}/checkpoints"

In [ ]:
%python

# Kafka-compatible endpoint of Event Hubs (port 9093). Username is always the literal "$ConnectionString".
jaas = (
  'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required '
  f'username="$ConnectionString" password="{event_hub_connection_string}";'
)

kafkaConf = {
  "kafka.bootstrap.servers": f"{eh_namespace}.servicebus.windows.net:9093",
  "subscribe": eh_name,
  "kafka.security.protocol": "SASL_SSL",
  "kafka.sasl.mechanism": "PLAIN",
  "kafka.sasl.jaas.config": jaas,
  "startingOffsets": "earliest",
  "maxOffsetsPerTrigger": 10000,
  "kafka.request.timeout.ms": "60000",
  "kafka.session.timeout.ms": "30000",
}

In [ ]:
%python

eventStreamDF = (spark.readStream
  .format("kafka")
  .options(**kafkaConf)
  .load()
)

eventStreamDF.printSchema()

In [ ]:
%python
from pyspark.sql.functions import col

# Kafka source exposes the payload as "value" (binary)
bodyDF = eventStreamDF.select(col("value").cast("STRING").alias("body"))

In [ ]:
%python
test_df = (spark.read
  .format("kafka")
  .options(**{**kafkaConf, "endingOffsets": "latest"})
  .load())

print(test_df.count())
display(test_df.select(col("value").cast("string")).limit(5))


In [ ]:
%python
import uuid
display(bodyDF, checkpointLocation=f"{checkpoint_base}/display_bodyDF_{uuid.uuid4().hex}", outputMode="append")

In [ ]:
from pyspark.sql.types import StructField, StructType, StringType, LongType, DoubleType

schema = StructType([
      StructField("MONTH", StringType(), True),
      StructField("DAY_OF_MONTH", StringType(), True),
      StructField("DAY_OF_WEEK", StringType(), True),
      StructField("OP_UNIQUE_CARRIER", StringType(), True),
      StructField("TAIL_NUM", StringType(), True),
      StructField("OP_CARRIER_FL_NUM", StringType(), True),
      StructField("ORIGIN_AIRPORT_ID", StringType(), True),
      StructField("ORIGIN", StringType(), True),
      StructField("ORIGIN_CITY_NAME", StringType(), True),
      StructField("DEST_AIRPORT_ID", StringType(), True),
      StructField("DEST", StringType(), True),
      StructField("DEST_CITY_NAME", StringType(), True),
      StructField("CRS_DEP_TIME", StringType(), True), 
      StructField("DEP_TIME", StringType(), True), 
      StructField("DEP_DELAY_NEW", StringType(), True), 
      StructField("DEP_DEL15", StringType(), True) ,
      StructField("DEP_TIME_BLK", StringType(), True), 
      StructField("CRS_ARR_TIME", StringType(), True) ,
      StructField("ARR_TIME", StringType(), True) ,
      StructField("ARR_DELAY_NEW", StringType(), True), 
      StructField("ARR_TIME_BLK", StringType(), True) ,
      StructField("CANCELLED", StringType(), True) ,
      StructField("CANCELLATION_CODE", StringType(), True) ,
      StructField("CRS_ELAPSED_TIME", StringType(), True) ,
      StructField("ACTUAL_ELAPSED_TIME", StringType(), True) ,
      StructField("DISTANCE", StringType(), True) ,
      StructField("DISTANCE_GROUP", StringType(), True) ,
      StructField("CARRIER_DELAY", StringType(), True) ,
      StructField("WEATHER_DELAY", StringType(), True) ,
      StructField("NAS_DELAY", StringType(), True) ,
      StructField("SECURITY_DELAY", StringType(), True) ,
      StructField("LATE_AIRCRAFT_DELAY", StringType(), True)
])

In [ ]:
%python

from pyspark.sql.functions import col, from_json

parsedEventsDF = bodyDF.select(
  from_json(col("body"), schema).alias("json"))

parsedEventsDF.printSchema()

In [ ]:
%python

flatSchemaDF = (parsedEventsDF
  .select(col("json.MONTH").alias("Month"),
          col("json.DAY_OF_MONTH").alias("Day_Of_Month"),
          col("json.DAY_OF_WEEK").alias("Day_Of_Week"),
          col("json.OP_UNIQUE_CARRIER").alias("OP_Unique_Carrier"),
          col("json.TAIL_NUM").alias("Tail_Num"),
          col("json.OP_CARRIER_FL_NUM").alias("OP_CARRIER_FL_NUM"),
          col("json.ORIGIN_AIRPORT_ID").alias("ORIGIN_AIRPORT_ID"),
          col("json.ORIGIN").alias("ORIGIN"),
          col("json.ORIGIN_CITY_NAME").alias("ORIGIN_CITY_NAME"),
          col("json.DEST_AIRPORT_ID").alias("DEST_AIRPORT_ID"),
          col("json.DEST").alias("DEST"),
          col("json.DEST_CITY_NAME").alias("DEST_CITY_NAME"),
          col("json.CRS_DEP_TIME").alias("CRS_DEP_TIME"),
          col("json.DEP_TIME").alias("DEP_TIME"),
          col("json.DEP_DELAY_NEW").alias("DEP_DELAY_NEW"),
          col("json.DEP_DEL15").alias("DEP_DEL15"),
          col("json.DEP_TIME_BLK").alias("DEP_TIME_BLK"),
          col("json.CRS_ARR_TIME").alias("CRS_ARR_TIME"),
          col("json.ARR_TIME").alias("ARR_TIME"),
          col("json.ARR_DELAY_NEW").alias("ARR_DELAY_NEW"),
          col("json.ARR_TIME_BLK").alias("ARR_TIME_BLK"),
          col("json.CANCELLED").alias("CANCELLED"),
          col("json.CANCELLATION_CODE").alias("CANCELLATION_CODE"),
          col("json.CRS_ELAPSED_TIME").alias("CRS_ELAPSED_TIME"),
          col("json.ACTUAL_ELAPSED_TIME").alias("ACTUAL_ELAPSED_TIME"),
          col("json.DISTANCE").alias("DISTANCE"),
          col("json.DISTANCE_GROUP").alias("DISTANCE_GROUP"),
          col("json.CARRIER_DELAY").alias("CARRIER_DELAY"),
          col("json.WEATHER_DELAY").alias("WEATHER_DELAY"),
          col("json.NAS_DELAY").alias("NAS_DELAY"),
          col("json.SECURITY_DELAY").alias("SECURITY_DELAY"),
          col("json.LATE_AIRCRAFT_DELAY").alias("LATE_AIRCRAFT_DELAY") )
)

In [ ]:
%python
# Optional preview only (runs until interrupted). Skip this cell before running the write cell below.
import uuid
display(flatSchemaDF, outputMode="append", checkpointLocation=f"{checkpoint_base}/display_flatSchemaDF_{uuid.uuid4().hex}")

In [ ]:
flatSchemaDF.createOrReplaceTempView("airlineEvents55")

Run the write cell below first, then query the Delta table (a streaming temp view can't be queried with `%sql` without a checkpoint).

In [ ]:
# Keep all columns: the DLT pipeline (3.dlt-batch-streaming.sql) needs fields like ORIGIN_CITY_NAME, DISTANCE, delays, etc.
final_df = flatSchemaDF

In [ ]:
%python
# Landing folder for the DLT pipeline (Auto Loader / read_files reads from here)
spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog}.{schema_name}.raw")
raw_path = f"/Volumes/{catalog}/{schema_name}/raw/airline22"

query = (final_df.writeStream
  .format("csv")
  .option("header", "true")
  .option("path", raw_path)
  .option("checkpointLocation", f"{checkpoint_base}/airline_events_csv")
  .outputMode("append")
  .trigger(availableNow=True)   # process everything currently in the event hub, then stop
  .start()
)
query.awaitTermination()

In [ ]:
%python
# Files written by the streaming query
display(dbutils.fs.ls(raw_path))

In [ ]:
%python
written_df = spark.read.option("header", "true").csv(raw_path)
display(written_df)
print(written_df.count())

In [ ]:
%sql
CREATE VOLUME IF NOT EXISTS airline_cluster.airline.raw;
